<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_99_optimization_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⚙️ **Module 99 — Optimization Deep Dive** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 14 · Theory & Responsibility (UDL deep-dives)


# Module 99 — Optimization Deep Dive

> Every training script picks an optimizer line near the top — and almost nobody questions it. This module is the **first-principles tour** of the optimizer zoo built around UDL Chapter 6 (Line Search · Gradient Descent · SGD · Momentum · Adam — 5 notebooks) + Chapter 7 (Backpropagation · Initialization — 3 notebooks), modernized to 2026: **AdamW · Lion · Schedule-Free · Shampoo · K-FAC · SOAP · SOPHIA · second-order · LR schedules · Xavier/He/LSUV/μP init**. By the end you'll know which optimizer matches which problem, how the modern frontier-LLM training recipes are tuned, and the math that connects them all.

### What you'll cover
1. **Line search** — the dumbest optimizer that still teaches everything
2. **GD → SGD → mini-batch** — the noise-vs-cost trade
3. **Momentum** — heavy-ball + Nesterov
4. **Adaptive methods** — AdaGrad → RMSProp → Adam → AdamW
5. **Modern adaptive** — Lion · Schedule-Free · Sophia · SOAP
6. **Second-order** — Newton · K-FAC · Shampoo · GGN
7. **Learning-rate schedules** — cosine, linear, warmup, WSD, cyclic
8. **Initialization** — Xavier · He · LSUV · μP (Maximum-Update Parameterization)
9. **Backpropagation revisited** — vector-Jacobian products, gradient checkpointing
10. The **2026 picker** — what frontier labs actually use, and why


## 1 · Line search — the warmup

Given a current point `x_t` and descent direction `d_t` (e.g. `-∇f(x_t)`), how far do you step?

**Exact line search:** find `α* = argmin_α f(x_t + α·d_t)` (closed-form only for quadratics).
**Backtracking line search (Armijo):** start with `α = 1`, halve until

```
f(x_t + α d_t) ≤ f(x_t) + c·α·∇f(x_t)·d_t        # c ~ 0.0001
```

**Strong Wolfe conditions** add a curvature term. Line search is **not used in deep learning training** (too expensive per step) — but it underpins L-BFGS (used for small-batch fine-tuning, temperature scaling in M98), Newton's method, and the theoretical convergence proofs of every other method here. UDL Chap6.1 walks the algorithm in 1D.

> 📐 **The intellectual purpose.** Every optimizer below is "line search but I'm cheating on the step size." SGD uses a hand-picked `lr`. Adam estimates per-parameter step sizes from gradient statistics. Newton's method uses a curvature-derived ideal step. Knowing line search makes the cheating visible.


## 2 · GD → SGD → mini-batch

**Gradient Descent (GD).** Step with the full-data gradient:
```
θ_{t+1} = θ_t − lr · (1/N) Σ_{i=1}^N ∇L_i(θ_t)
```
Convex convergence at `O(1/t)`; strongly convex at `O(c^t)`. **Impossible in deep learning** — N=10M means each step touches 10M examples.

**Stochastic Gradient Descent (SGD).** Step with the gradient at a single random example:
```
θ_{t+1} = θ_t − lr · ∇L_{i_t}(θ_t)
```
Same expected gradient, much higher variance. Convergence `O(1/√t)` to a noise-floor proportional to `lr·σ²`.

**Mini-batch SGD.** The sweet spot — batch of B examples, `O(1/√t)` convergence with `1/B` variance reduction. Modern training uses `B ∈ [256, 4096]` for vision; `B ∈ [1M, 16M tokens]` for LLM pretraining.

| Method | Memory | Gradient noise | Step rate |
|---|---|---|---|
| Full GD | O(N) per step | None | Slow (1 step/full pass) |
| SGD (B=1) | O(1) | Very high | Fast (N steps/pass) |
| Mini-batch (B=256-4096) | O(B) | Moderate | Sweet spot |

**Implicit benefit of SGD noise.** Section 6.3 of UDL shows experimentally that SGD escapes saddle points (in high D, almost every critical point is a saddle, M98 callback) and finds **flatter minima** than full GD — which generalize better.


## 3 · Momentum — heavy ball + Nesterov

Plain SGD oscillates in ravines. **Polyak's heavy-ball momentum** adds a velocity term:

```
v_{t+1} = β · v_t + (1−β) · ∇L(θ_t)        # β ~ 0.9
θ_{t+1} = θ_t − lr · v_{t+1}
```

The exponential average smooths the gradient signal and accelerates flat directions. **`β=0.9` is the universal default;** `0.99` for very noisy losses.

**Nesterov accelerated gradient (Nesterov 1983):**
```
v_{t+1} = β · v_t + ∇L(θ_t + β · v_t · lr)   # gradient at the look-ahead point
θ_{t+1} = θ_t − lr · v_{t+1}
```

Theoretically `O(1/t²)` vs heavy-ball `O(1/t)` on convex functions. In deep learning the difference is small (4-7% in practice), but **Nesterov is the standard for SGD-with-momentum** in modern CV training (timm uses it).

> 🧪 **Momentum as second-moment information.** Heavy ball implicitly estimates the *direction* the loss surface is curving. It's a first step toward second-order methods — keep the trajectory information of where the gradient was pointing, not just where it points now.


## 4 · Adaptive learning rates — AdaGrad → Adam → AdamW

**AdaGrad** (Duchi 2011). Per-parameter learning rate inversely proportional to historical gradient magnitudes:

```
G_t = G_{t-1} + (∇L_t)²                    # element-wise
θ_{t+1} = θ_t − lr · ∇L_t / (√G_t + ε)
```

Great for sparse features (each feature gets its own LR). Fails on dense problems because `G_t` keeps growing → LR → 0.

**RMSProp** (Hinton lecture 2012) — same idea with exponential decay:
```
G_t = β · G_{t-1} + (1−β)·(∇L_t)²
```

**Adam** (Kingma & Ba 2015) = momentum + RMSProp:
```
m_t = β1 · m_{t-1} + (1−β1) · ∇L_t            # first moment
v_t = β2 · v_{t-1} + (1−β2) · (∇L_t)²         # second moment
m̂_t = m_t / (1 − β1^t)                         # bias correction
v̂_t = v_t / (1 − β2^t)
θ_{t+1} = θ_t − lr · m̂_t / (√v̂_t + ε)
```

**`β1=0.9, β2=0.999, ε=1e-8` is the universal default.** Adam revolutionized training NLP, RL, GANs — anywhere non-stationary or sparse gradients were a problem.

**AdamW** (Loshchilov 2019 — M98 callback). Adam + L2 in the loss doesn't actually shrink weights; AdamW *decouples* weight decay:
```
θ_{t+1} = θ_t − lr · (m̂_t / (√v̂_t + ε)) − lr · λ · θ_t       # explicit decoupled decay
```

**Every 2026 production training recipe uses AdamW.** Defaults: `lr ~ 3e-4 (ViT, Transformer), 1e-3 (CNN), 1e-5 to 5e-5 (LLM fine-tune); wd ~ 0.01-0.1`.


In [ ]:
import torch

@torch.no_grad()
def adamw_step(params, grads, state, lr=3e-4, b1=0.9, b2=0.999,
               eps=1e-8, wd=0.01, t=1):
    for p, g in zip(params, grads):
        s = state.setdefault(id(p), {"m": torch.zeros_like(p), "v": torch.zeros_like(p)})
        s["m"].mul_(b1).add_(g, alpha=1 - b1)
        s["v"].mul_(b2).addcmul_(g, g, value=1 - b2)
        m_hat = s["m"] / (1 - b1 ** t)
        v_hat = s["v"] / (1 - b2 ** t)
        p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
        p.mul_(1 - lr * wd)                                # decoupled WD


## 5 · The 2024-2026 wave — Lion, Schedule-Free, Sophia, SOAP

| Optimizer | Year | Idea | Where it shines |
|---|---|---|---|
| **Lion** (Chen, EvoLved Sign Momentum) | 2023 | Drop `v_t`; use `sign(momentum)` for update; ~½ memory of Adam | Vision (EfficientNet-V2, ConvNeXt-V2); slightly under Adam on LLMs |
| **Schedule-Free** (Defazio et al.) | 2024 | Eliminates the LR schedule entirely; uses exponential averaging of iterates internally | Pre-training (LLM, vision) where you don't want to tune total-step count |
| **Sophia** (Liu et al., Second-Order CHubin/HuberApprox) | 2023 | Estimates diagonal Hessian via Hutchinson; clips Newton steps | LLM pretraining — ~2× speedup over AdamW in early training |
| **SOAP** (Vyas, Morwani, Anil 2024) | 2024 | Shampoo + Adam: precondition Adam state with low-rank Shampoo matrices | LLM pretraining — frontier-lab winner in 2024-25 |
| **CAME** (Confidence-guided Adaptive Memory-Efficient) | 2024 | AdaFactor + variance estimation; low-mem | TPU pretraining |
| **AdaFactor** (Shazeer 2018) | 2018 | Factored second moment (row × col only); 1/√d memory of Adam | T5 / PaLM / LaMDA / Gemini — Google default |
| **LAMB** (You et al. 2020) | 2020 | Layer-wise adaptive LR; enables huge batches (32k+) for BERT | Pre-training at extreme batch sizes |

**Lion is half memory of Adam** — for a 7B model that's 14GB → 7GB of optimizer state. Lion's sign-of-momentum update plays well with adversarial training and image classification. It underperforms Adam by ~0.5% on most LLM tasks, so frontier labs still default to AdamW.

**SOAP and Sophia are the frontier of 2024-2026 pretraining.** Combining Shampoo's preconditioner with Adam's adaptivity gives a 30-40% wall-clock speedup over AdamW on GPT-scale pretraining. NVIDIA's Nemotron-4, Meta's Llama-3.1+, and several frontier labs use SOAP variants.

> 🧱 **Memory is the optimizer's real cost.** A 70B model trained with AdamW needs ~140GB of optimizer state (vs. 140GB weights). ZeRO-1/2/3 (DeepSpeed, FSDP — M66) shards this across devices. **AdaFactor and Lion exist primarily to reduce optimizer-state memory.**


## 6 · Second-order methods — Newton, K-FAC, Shampoo, GGN

**Newton's method:** `θ_{t+1} = θ_t − H_t⁻¹ · g_t`. For a quadratic, Newton converges in **one step**. For a deep net, `H` is `[P × P]` — impossible to store, much less invert, for `P=1B+`.

| Method | Approximation | Cost |
|---|---|---|
| **L-BFGS** | Limited-memory inverse-Hessian approx via gradient history | Practical for B≤30 history; used in M98 temperature scaling |
| **K-FAC** (Martens 2015) | Kronecker-factored approximation of layer-wise Fisher: `H_l ≈ A_l ⊗ G_l` | ~1.5-3× per-step cost, often ~2× fewer steps |
| **Shampoo** (Gupta 2018) | K-FAC's generalization to tensors of any rank; per-tensor preconditioner | Strong on vision, Transformer; SOAP's base |
| **GGN (Generalized Gauss-Newton)** | `H ≈ J^T · diag · J`; positive semidefinite by construction | Foundation of K-FAC, Sophia |
| **Distributed Shampoo (Anil 2020)** | Block-diagonal Shampoo across data-parallel workers | Used in PaLM, internal Google training |
| **Sophia** (2023) | Diagonal Hessian via Hutchinson + step clipping | LLM-friendly; ~2× speedup |

**The fundamental trade-off:** second-order methods take fewer steps but each step is more expensive. The break-even depends on (a) how poorly first-order methods condition the problem (transformers are particularly bad), (b) compute budget for matrix factorization, (c) memory headroom for the preconditioner.

> 🧮 **Why frontier-LLM labs care.** A 30% step-count reduction × 100k-1M-step pretraining = millions of dollars saved per run. SOAP and Sophia exist because Anthropic, DeepMind, Meta, and OpenAI are all pushing on this — even a 5% wall-clock improvement is enormous at scale.


## 7 · Learning-rate schedules

The LR schedule matters as much as the optimizer. Six families:

| Schedule | Shape | Use |
|---|---|---|
| **Constant** | Flat | Online RL, fine-tuning small batches |
| **Step decay** | Drop by 10× at fixed epochs | Classic CNN training (ResNet, AlexNet) |
| **Cosine** (Loshchilov 2017) | Half-cycle cosine from `lr_max` to ~0 | Modern vision (timm default) + LLM SFT |
| **Linear** | Linear from `lr_max` to 0 | LLM pretraining (Llama, GPT-3) — empirically slightly better than cosine for next-token loss |
| **WSD (Warmup-Stable-Decay)** (Hu 2024 MiniCPM) | Warmup → constant for most of training → fast decay at end | LLM pretraining where you may extend training later |
| **Cyclic / OneCycle** (Smith 2017) | Triangular cycles between min and max | Super-convergence; small-data CV |
| **Schedule-Free** (Defazio 2024) | None — averaging replaces it | Removes the "total steps" hyperparameter |

**Warmup** (linear ramp-up of LR over the first ~1-5% of steps) is **mandatory for transformers** (Liu 2020). Without warmup, AdamW's per-parameter LRs are too aggressive in the first few hundred steps and the model collapses. The standard recipe:

```
LR(step) = warmup(0 → max, over W steps) + main_schedule(max → 0, over T−W steps)
```

`W ≈ 2000` for BERT-base; `W ≈ 8000` for Llama-3-70B; longer for sparser models.

> 📈 **Why WSD is winning in 2024-2026.** Pretraining LLMs is computationally enormous; you'd like to extend training if you have more budget. Cosine requires committing to a total step count upfront. WSD lets you decide "actually let's train 30% longer" and just delay the decay phase. MiniCPM, Phi-3, Gemma-2 all use WSD-style schedules.


## 8 · Initialization — Xavier · He · LSUV · μP

Bad initialization can make training fail entirely. Each layer of weights needs a variance scaled so that **forward activations and backward gradients have stable magnitude across depth**.

| Method | Variance | When |
|---|---|---|
| **Xavier / Glorot** (2010) | `Var(W) = 2 / (n_in + n_out)` | tanh / sigmoid activations |
| **He / Kaiming** (2015) | `Var(W) = 2 / n_in` | ReLU / GELU activations (default for CNN, Transformer FFN) |
| **LSUV** (Layer-Sequential Unit-Variance, Mishkin 2015) | Orthogonal init + data-dependent scale until activations have unit variance | Very deep networks; residual nets pre-BN |
| **Iterative re-init** (Fixup, ReZero) | Initialize residual blocks to identity; learn the residual mix | Train very deep nets without BN |
| **μP (Yang et al. 2021)** | Re-scale each layer so transferring hyperparameters from small to large model **works** | Frontier-LLM scaling; used by OpenAI, Microsoft |

**μP (Maximum-Update Parameterization)** is the modern breakthrough. Standard init's optimal LR shifts as width grows; μP rescales weights so that **the optimal LR is the same regardless of width**. You can tune hyperparameters on a tiny 50M-parameter network and the LR / wd / β2 transfers exactly to a 70B model. This is how frontier labs avoid spending compute tuning hyperparameters at scale.

> 🪜 **He for everything ReLU-family, Xavier for tanh/sigmoid, μP if you're training a really big model**. Use the default initializer in your framework and don't worry; reach for LSUV if you're training >100 layers without normalization; reach for μP if you're sweeping hyperparameters across model scales.


## 9 · Backprop revisited — VJPs, gradient checkpointing, mixed precision

UDL Chap7.1 + 7.2 build backprop from scratch in a toy 2-layer model. The two industrially-relevant lessons:

**Vector-Jacobian product (VJP).** Modern autodiff (PyTorch, JAX) doesn't compute the full Jacobian — it computes `v^T · J` by chaining VJPs through the computation graph. This is **memory-linear in graph size**, not quadratic.

**Gradient checkpointing.** Store only a subset of activations in the forward pass; recompute the rest in the backward pass. Trades **~30% extra compute for ~50% less memory** — crucial for fine-tuning large models. PyTorch `torch.utils.checkpoint.checkpoint`, HuggingFace `gradient_checkpointing=True`.

**Mixed-precision training (bf16/fp16).** Compute forward and backward in lower precision; keep a master copy of weights in fp32 for stability. **bf16 is the modern default** (same dynamic range as fp32, half the memory and 2× throughput on modern hardware). Used in every LLM training pipeline since 2022.

**Loss scaling (fp16-specific).** Multiply the loss by a large scalar before backward to prevent underflow of gradients; divide gradients back before the optimizer step. Not needed for bf16 because of the wider exponent range.

> 🛠️ **Production training stack (2026).** AdamW (or SOAP) + cosine/linear schedule + 5% warmup + bf16 mixed-precision + gradient checkpointing + ZeRO-3 or FSDP sharding (M66). That single combination is behind every LLaMA, Mistral, Qwen, DeepSeek pretraining run published openly.


## 10 · The 2026 picker — what frontier labs actually use

```
                              Training a deep model?
                                    │
                ┌──────────────────┼─────────────────┐
              vision /             LLM                generic
              vision-LM       (transformer)             (RL, GAN, ...)
                │                    │                       │
            AdamW                AdamW (default)          Adam / AdamW
            or Lion              SOAP (frontier)
            cosine lr            cosine or WSD lr
            label smooth         linear warmup
            EMA weights          ZeRO-3 / FSDP
            bf16                 bf16 + grad checkpointing
```

**Frontier labs (May 2026):**
- **OpenAI** — AdamW with custom schedule; μP for hyperparameter transfer
- **Anthropic** — AdamW; gradient checkpointing; ZeRO-1 → ZeRO-3 schedule
- **DeepMind** — AdaFactor (Gemini) + custom schedule; second-order experiments
- **Meta** — AdamW + cosine; SOAP variants in Llama-3.5+ research
- **Mistral** — AdamW + linear schedule with WSD-style extensions

**Open-source SFT / fine-tuning (TRL, Axolotl, Unsloth):**
- AdamW 8-bit (bitsandbytes) — 25% the memory of fp32 AdamW
- LoRA / QLoRA — most parameters frozen; only a few millions trained
- Schedule: cosine with 5-10% warmup
- LR: `1e-5` to `5e-5` for full fine-tune; `1e-4` to `5e-4` for LoRA

> 🎓 **The takeaway.** The optimizer zoo is large but the *production frontier* is narrow: **AdamW + cosine schedule + bf16** trains 95% of modern models. The 5% that justify SOAP / Sophia / Lion / μP are the ones spending millions on pretraining. Everyone else should know the menu but pick from the top 3.

## ✅ Recap

- **Line search** is the theoretical primitive; everyone "cheats" on the step size differently.
- **GD → SGD → mini-batch** — SGD noise is a feature (escapes saddles, finds flat minima).
- **Momentum** (β=0.9) smooths trajectories; Nesterov adds look-ahead.
- **Adam** = momentum + per-parameter RMSProp; **AdamW** decouples weight decay.
- **2024-2026 wave**: **Lion** (½ memory) · **Schedule-Free** (no LR schedule) · **Sophia** (diagonal Hessian) · **SOAP** (Shampoo + Adam) · **AdaFactor** (1/√d memory).
- **Second-order** (K-FAC, Shampoo, SOAP, Sophia) reduces step count 30-50% but each step is heavier — economic at frontier scale.
- **LR schedule**: cosine for SFT, linear for LLM pretrain, WSD for extensible training; mandatory warmup for transformers.
- **Init**: He for ReLU, Xavier for tanh, **μP** for hyperparameter transfer across scales.
- **Backprop tricks**: VJPs, gradient checkpointing, bf16 mixed-precision, loss scaling.
- **2026 production frontier**: AdamW + cosine + bf16 + ZeRO/FSDP. Reach for SOAP only at frontier scale.

Next: **M100 — Fairness · Bias Mitigation · Explainability** (course completion).
